In [1]:
!pip install transformers torch pandas matplotlib openpyxl tqdm emoji -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 5.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re
import emoji
import torch
import warnings
import matplotlib.pyplot as plt

from datetime import datetime
from tqdm import tqdm
from transformers import pipeline as hf_pipeline

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

warnings.filterwarnings('ignore')
tqdm.pandas()

In [3]:
CONFIG = {

    # INPUT / OUTPUT
    'input_file'      : '/test_data.xlsx,
    'output_file'     : '/content/results.xlsx',
    'eda_output'      : '/content/eda.png',

    # MODEL RULES
    'min_words'       : 4,
    'irony_threshold' : 0.80,

    # BART CONFIDENCE
    'bart_high'       : 0.60,
    'bart_medium'     : 0.35,

    # SARCASM
    'sarc_score_min'  : 0.20,
}

# =============================================================================
# LABELS
# =============================================================================

EMOTION_LABELS = [
    'frustrated or angry',
    'hopeful or relieved',
    'fearful or worried',
    'demanding action',
    'informed reporting',
    'sarcastic criticism',
]

LABEL_TO_SENTIMENT = {

    'frustrated or angry' : 'Negative',
    'fearful or worried'  : 'Negative',
    'demanding action'    : 'Negative',
    'sarcastic criticism' : 'Negative',
    'hopeful or relieved' : 'Positive',
    'informed reporting'  : 'Neutral',
}

ROBERTA_MAP = {

    'positive' : 'Positive',
    'negative' : 'Negative',
    'neutral'  : 'Neutral',
}

ICON = {

    'Positive': '🟢',
    'Negative': '🔴',
    'Neutral': '🟡'
}

# =============================================================================
# REGEX RULES
# =============================================================================

# SHRINKFLATION
SHRINKFLATION = re.compile(

    r'\b(?:weight|size|quantity|volume|amount|content)\s+reduced\b'
    r'|\breduced from \d+\s*\w+ to \d+\s*\w+\b'
    r'|\b(?:kg|g|ml|l)\s+to\s+\d+\s*(?:kg|g|ml|l)\b',

    re.IGNORECASE
)

# HEDGE / SPECULATIVE
HEDGE_PATTERN = re.compile(

    r'\b(may|might|could|expected to|hopes? to|likely to|possibly|potential(?:ly)?)\b',

    re.IGNORECASE
)

# RESOLUTION PHRASES
RESOLUTION_PHRASES = re.compile(

    r'\b(war (?:is )?(?:finally )?over|'
    r'ceasefire (?:signed|holding|reached|announced)|'
    r'(?:prices?|costs?) (?:have been |finally )?reduced|'
    r'great relief|'
    r'debt waiver|'
    r'finally (?:reduced|provided|stabilis)|'
    r'supply (?:has )?improved|'
    r'subsidy (?:provided|announced|scheme)|'
    r'energy markets? (?:beginning to )?stabilis)\b',
    re.IGNORECASE
)

# STATISTICAL FACTS
STAT_PATTERN = re.compile(
    r'(\d+\s*(?:percent|%|degrees?))'
    r'|(\baccording to\b|\brecorded at\b|\bdata shows?\b|\breport(?:s|ed)?\b)',
    re.IGNORECASE
)

# NEGATION RULES
NEGATION_PAIRS = [
    (r'\bcannot be denied that\b', 'it is confirmed that'),
    (r'\bhas not\s+(?:eased|improved|resolved|recovered)\b',
     'has remained bad'),
    (r'\bnot (?:eased|improved|resolved|stabilised|stabilized)\b',
     'still bad'),
    (r'\bnot (?:getting |becoming )?(?:worse|worsening)\b',
     'improving'),
    (r'\bno (?:longer )?improving\b',
     'worsening'),
    (r'\bnot (?:deteriorating|worsening)\b',
     'stable'),
]

print("Checking device...")

device = 0 if torch.cuda.is_available() else -1

if device == 0:
    print("✅ GPU ENABLED")
else:
    print("⚠️ Running on CPU")

Checking device...
⚠️ Running on CPU


In [4]:
print("\nLoading models...")

BART = hf_pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=device
)
print("✅ BART loaded")

ROBERTA = hf_pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    device=device
)
print("✅ RoBERTa loaded")

IRONY = hf_pipeline(
    'text-classification',
    model='cardiffnlp/twitter-roberta-base-irony',
    device=device
)
print("✅ Irony model loaded")


Loading models...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-irony
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

✅ Irony model loaded


In [5]:
def clean(text):
    if not isinstance(text, str):
        return ''

    text = text.strip()

    # Remove URLs
    text = re.sub(r'http\S+', '', text)

    # Remove hashtags but keep words
    text = re.sub(r'#(\w+)', r'\1', text)

    # Remove mentions
    text = re.sub(r'@\w+', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# =============================================================================
# NEGATION NORMALIZER
# =============================================================================

def normalise_negation(text):

    for pattern, replacement in NEGATION_PAIRS:

        text = re.sub(
            pattern,
            replacement,
            text,
            flags=re.IGNORECASE
        )

    return text

# =============================================================================
# STATISTICAL FACT DETECTOR
# =============================================================================

def is_statistical_fact(text):

    OPINION_WORDS = re.compile(

        r'\b(great|terrible|disaster|amazing|awful|crisis|'
        r'shocking|wonderful|finally|unfortunately|'
        r'brilliant|ridiculous|cannot afford|destroying)\b',

        re.IGNORECASE
    )

    has_stat = bool(STAT_PATTERN.search(text))

    has_opinion = bool(OPINION_WORDS.search(text))

    word_count = len(text.split())

    return has_stat and not has_opinion and word_count <= 25

# =============================================================================
# ENSEMBLE LOGIC
# =============================================================================

def ensemble(bart_sent, bart_conf, roberta_sent):

    # Both agree
    if bart_sent == roberta_sent:
        return bart_sent

    # BART high confidence
    if bart_conf == 'HIGH':
        return bart_sent

    # Medium confidence
    if bart_conf == 'MEDIUM':

        if 'Neutral' in (bart_sent, roberta_sent):
            return 'Neutral'

        return roberta_sent

    # Low confidence
    return roberta_sent


In [6]:
def predict(text):

    cleaned = clean(text)

    # TOO SHORT
    if not cleaned or len(cleaned.split()) < CONFIG['min_words']:

        return (
            'Neutral',
            'informed reporting',
            0.0,
            False,
            'LOW'
        )

    # STATISTICAL FACT
    if is_statistical_fact(cleaned):

        return (
            'Neutral',
            'informed reporting',
            1.0,
            False,
            'HIGH'
        )

    # NEGATION NORMALIZATION
    normalised = normalise_negation(cleaned)

    # =========================================================================
    # BART
    # =========================================================================

    zs = BART(
        normalised[:512],
        candidate_labels=EMOTION_LABELS
    )

    label_scores = dict(zip(zs['labels'], zs['scores']))
    top_label = zs['labels'][0]
    top_score = zs['scores'][0]
    sarc_score = label_scores.get(
        'sarcastic criticism',
        0
    )

    relieved_score = label_scores.get(
        'hopeful or relieved',
        0
    )

    bart_sent = LABEL_TO_SENTIMENT.get(
        top_label,
        'Neutral'
    )

    # CONFIDENCE
    if top_score >= CONFIG['bart_high']:
        bart_conf = 'HIGH'

    elif top_score >= CONFIG['bart_medium']:
        bart_conf = 'MEDIUM'

    else:
        bart_conf = 'LOW'

    # =========================================================================
    # ROBERTA
    # =========================================================================

    rb_raw = ROBERTA(normalised[:512])[0]

    roberta_sent = ROBERTA_MAP.get(
        rb_raw['label'].lower(),
        'Neutral'
    )

    # =========================================================================
    # IRONY
    # =========================================================================

    ir = IRONY(cleaned[:512])[0]

    is_irony = (

        ir['label'] == 'irony'
        and
        ir['score'] >= CONFIG['irony_threshold']
    )

    # =========================================================================
    # RULE FLAGS
    # =========================================================================

    is_hedged = bool(
        HEDGE_PATTERN.search(normalised)
    )

    has_explicit_resolution = bool(
        RESOLUTION_PHRASES.search(normalised)
    )

    # =========================================================================
    # RULE 1 — SHRINKFLATION
    # =========================================================================

    if SHRINKFLATION.search(normalised):

        return (
            'Negative',
            'frustrated or angry',
            top_score,
            False,
            'MEDIUM'
        )

    # =========================================================================
    # RULE 2 — RESOLUTION PHRASES
    # =========================================================================

    if (

        has_explicit_resolution
        and
        sarc_score <= CONFIG['sarc_score_min']
    ):

        if is_hedged:

            return (
                'Neutral',
                'informed reporting',
                top_score,
                False,
                'MEDIUM'
            )

        return (
            'Positive',
            'hopeful or relieved',
            top_score,
            False,
            'MEDIUM'
        )

    # =========================================================================
    # RULE 3 — HEDGED
    # =========================================================================

    if is_hedged:

        return (
            'Neutral',
            'informed reporting',
            top_score,
            False,
            'MEDIUM'
        )

    # =========================================================================
    # RULE 4 — SARCASM
    # =========================================================================

    if sarc_score > CONFIG['sarc_score_min']:

        return (
            'Negative',
            'sarcastic criticism',
            sarc_score,
            False,
            'LOW'
        )

    # =========================================================================
    # RULE 5 — IRONY
    # =========================================================================

    if is_irony:

        return (
            'Negative',
            'sarcastic criticism',
            top_score,
            True,
            'MEDIUM'
        )

    # =========================================================================
    # RULE 6 — NEGATION CONFIRMATION
    # =========================================================================

    NEGATION_CONFIRMED = re.compile(

        r'\b(has remained bad|still bad|worsening|'
        r'confirmed that.*bad|shortage.*remains|'
        r'situation.*bad|not improving)\b',

        re.IGNORECASE
    )

    if NEGATION_CONFIRMED.search(normalised):

        return (
            'Negative',
            'frustrated or angry',
            top_score,
            False,
            'MEDIUM'
        )

    # =========================================================================
    # RULE 7 — ENSEMBLE
    # =========================================================================

    final = ensemble(

        bart_sent,
        bart_conf,
        roberta_sent
    )

    # =========================================================================
    # RULE 8 — CONTEXT FLIP GUARD
    # =========================================================================

    if (

        bart_sent == 'Positive'
        and
        roberta_sent == 'Negative'
        and
        relieved_score < 0.25
    ):

        final = 'Negative'

        bart_conf = 'LOW'

    return (

        final,
        top_label,
        round(top_score, 3),
        is_irony,
        bart_conf
    )

# =============================================================================
# COLUMN DETECTOR
# =============================================================================

def detect_columns(df):

    columns = [c.lower() for c in df.columns]

    comment_col = None
    date_col = None
    type_col = None

    # COMMENT COLUMN
    for c in df.columns:

        lc = c.lower()

        if any(
            x in lc for x in
            ['comment', 'text', 'tweet', 'sentence', 'review']
        ):
            comment_col = c
            break

    # DATE COLUMN
    for c in df.columns:

        lc = c.lower()

        if any(
            x in lc for x in
            ['date', 'time', 'created']
        ):
            date_col = c
            break

    # PLATFORM COLUMN
    for c in df.columns:

        lc = c.lower()

        if any(
            x in lc for x in
            ['platform', 'source', 'type']
        ):
            type_col = c
            break

    if comment_col is None:
        comment_col = df.columns[0]

    print(f"✅ Comment column : {comment_col}")

    if date_col:
        print(f"✅ Date column    : {date_col}")

    if type_col:
        print(f"✅ Type column    : {type_col}")

    return comment_col, date_col, type_col

In [7]:
def plot_eda(df, sentiment_col='sentiment',
             date_col=None, type_col=None):

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    fig.suptitle(
        'Sentiment EDA',
        fontsize=14,
        fontweight='bold'
    )

    colors = {

        'Positive':'mediumseagreen',
        'Negative':'coral',
        'Neutral':'gold'
    }

    # =========================================================================
    # SENTIMENT DISTRIBUTION
    # =========================================================================

    counts = df[sentiment_col].value_counts()

    bars = axes[0].bar(

        counts.index,
        counts.values,

        color=[
            colors.get(l, 'grey')
            for l in counts.index
        ]
    )

    axes[0].set_title('Sentiment Distribution')

    for bar, val in zip(bars, counts.values):

        axes[0].text(

            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3,

            str(val),

            ha='center',
            fontweight='bold'
        )

    # =========================================================================
    # PLATFORM WISE
    # =========================================================================

    if type_col and type_col in df.columns:

        cross = pd.crosstab(

            df[type_col],
            df[sentiment_col]
        )

        cross.plot(

            kind='bar',
            ax=axes[1],

            color=[
                colors.get(c, 'grey')
                for c in cross.columns
            ]
        )

        axes[1].set_title('Sentiment by Platform')

        axes[1].tick_params(axis='x', rotation=0)

        axes[1].legend(loc='upper right')

    else:
        axes[1].axis('off')

    # =========================================================================
    # OVER TIME
    # =========================================================================

    if date_col and date_col in df.columns:

        try:

            df[date_col] = pd.to_datetime(
                df[date_col],
                errors='coerce'
            )

            t = df.groupby([
                df[date_col].dt.to_period('M'),
                sentiment_col
            ]).size().unstack(fill_value=0)

            t.plot(ax=axes[2])

            axes[2].set_title('Sentiment Over Time')

            axes[2].tick_params(axis='x', rotation=45)

        except:

            axes[2].axis('off')

    else:
        axes[2].axis('off')

    plt.tight_layout()

    plt.savefig(

        CONFIG['eda_output'],

        dpi=150,
        bbox_inches='tight'
    )

    plt.show()

    print(f"\nEDA saved → {CONFIG['eda_output']}")


In [8]:
def export(df, path):

    wb = Workbook()

    ws = wb.active

    ws.title = "Results"

    cols = [c for c in df.columns]

    hfill = PatternFill(

        "solid",

        start_color="1a1a2e",
        end_color="1a1a2e"
    )

    hfont = Font(

        bold=True,
        color="FFFFFF",
        name="Calibri",
        size=10
    )

    cmap = {

        'Positive':'C8F7C5',
        'Negative':'FADBD8',
        'Neutral':'FFF9C4'
    }

    # HEADERS
    for ci, col in enumerate(cols, 1):

        cell = ws.cell(

            row=1,
            column=ci,
            value=col.upper()
        )

        cell.font = hfont

        cell.fill = hfill

        cell.alignment = Alignment(

            horizontal='center',
            wrap_text=True
        )

        ws.column_dimensions[
            get_column_letter(ci)
        ].width = 60 if 'comment' in col.lower() else 20

    ws.freeze_panes = "A2"

    # ROWS
    for ri, row in df.iterrows():

        er = ri + 2

        sent = row.get('sentiment', 'Neutral')

        rfill = PatternFill(

            "solid",

            start_color=cmap.get(sent, 'FFFFFF'),
            end_color=cmap.get(sent, 'FFFFFF')
        )

        for ci, val in enumerate(row, 1):

            cell = ws.cell(

                row=er,
                column=ci,
                value=val
            )

            cell.font = Font(

                name="Calibri",
                size=9
            )

            cell.fill = rfill

            cell.alignment = Alignment(

                wrap_text=True,
                vertical='top'
            )

        ws.row_dimensions[er].height = 35

    # =========================================================================
    # SUMMARY SHEET
    # =========================================================================

    ws2 = wb.create_sheet("Summary")

    for row in [

        ["SENTIMENT RESULTS"],

        [''],

        ["Total comments", len(df)],

        ["Positive",
         int(df['sentiment'].value_counts().get('Positive',0))],

        ["Negative",
         int(df['sentiment'].value_counts().get('Negative',0))],

        ["Neutral",
         int(df['sentiment'].value_counts().get('Neutral',0))],

        [''],

        ["Irony detected",
         int(df['irony'].sum())
         if 'irony' in df.columns else 'N/A'],

        ["Processed at",
         datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
    ]:

        ws2.append(row)

    ws2['A1'].font = Font(
        bold=True,
        size=13
    )

    ws2.column_dimensions['A'].width = 25
    ws2.column_dimensions['B'].width = 20

    wb.save(path)

    print(f"\nResults saved → {path}")


In [9]:
def run_pipeline():

    print(f"\n{'='*60}")
    print("ADVANCED SENTIMENT ANALYSIS PIPELINE")
    print(f"Started : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*60}")

    # =========================================================================
    # LOAD FILE
    # =========================================================================

    f = CONFIG['input_file']

    if f.endswith('.xlsx'):
        df = pd.read_excel(f)
    else:
        df = pd.read_csv(f)

    print(f"\nLoaded : {len(df)} rows")

    # =========================================================================
    # DETECT COLUMNS
    # =========================================================================

    print("\nDetecting columns...")

    comment_col, date_col, type_col = detect_columns(df)

    # =========================================================================
    # PREDICTIONS
    # =========================================================================

    print(f"\nAnalysing {len(df)} comments...")

    preds = df[comment_col].progress_apply(

        lambda x: pd.Series(

            predict(x),

            index=[
                'sentiment',
                'emotion',
                'score',
                'irony',
                'confidence'
            ]
        )
    )

    df = pd.concat([df, preds], axis=1)

    # =========================================================================
    # PRINT RESULTS
    # =========================================================================

    print(f"\n{'─'*50}")
    print("RESULTS")
    print(f"{'─'*50}")

    for _, row in df.iterrows():

        icon = ICON.get(row['sentiment'], '🟡')

        print(

            f"{icon} "
            f"{row['sentiment']:10} "
            f"({row['confidence']:6}) | "
            f"{str(row[comment_col])[:65]}"
        )

    print("\nSentiment Distribution:")

    print(df['sentiment'].value_counts())

    # =========================================================================
    # EDA
    # =========================================================================

    plot_eda(

        df,

        'sentiment',

        date_col,

        type_col
    )

    # =========================================================================
    # EXPORT
    # =========================================================================

    export(

        df,

        CONFIG['output_file']
    )

    print(f"\n{'='*60}")
    print(f"DONE — {len(df)} comments processed")
    print(f"{'='*60}")

    return df


In [10]:
def check_comment():

    print("\n" + "="*60)
    print("LIVE COMMENT CHECKER")
    print("="*60)

    while True:

        text = input("\nEnter comment (or type 'exit'): ")

        if text.lower() == 'exit':
            break

        sent, emotion, score, irony, conf = predict(text)

        icon = ICON.get(sent, '🟡')

        print("\n" + "-"*50)

        print(f"Comment      : {text}")
        print(f"Sentiment    : {icon} {sent}")
        print(f"Emotion      : {emotion}")
        print(f"Score        : {score}")
        print(f"Irony        : {irony}")
        print(f"Confidence   : {conf}")

        print("-"*50)

In [11]:
def quick_test():

    tests = [

        # POSITIVE
        (
            "for LPG prices have been reduced this month, great relief families",
            "Positive"
        ),

        (
            "War finally over as ceasefire signed between Israel and Hamas",
            "Positive"
        ),

        (
            "Farmer debt waiver scheme announced covering all outstanding loans",
            "Positive"
        ),

        # NEGATIVE
        (
            "LPG cylinder weight reduced from 14kg to 12kg this quarter",
            "Negative"
        ),

        (
            "Oh great, LPG prices went up again. Just what we needed 👏",
            "Negative"
        ),

        (
            "Brilliant move cutting farmer subsidies during a food crisis",
            "Negative"
        ),

        # NEUTRAL
        (
            "India imports 85 percent of its oil needs from the middle east",
            "Neutral"
        ),

        (
            "Heatwave temperature recorded at 47 degrees in Delhi yesterday",
            "Neutral"
        ),
    ]

    print(f"\n{'#':<4} {'Expected':<10} {'Got':<10} {'Match':<6}")

    print("─" * 70)

    correct = 0

    for i, (text, expected) in enumerate(tests, 1):

        sent, label, score, irony, conf = predict(text)

        match = "✓" if sent == expected else "✗"

        if sent == expected:
            correct += 1

        print(

            f"{i:<4} "
            f"{expected:<10} "
            f"{sent:<10} "
            f"{match:<6}"
        )

    print(
        f"\nAccuracy : "
        f"{correct}/{len(tests)} = "
        f"{correct/len(tests)*100:.0f}%"
    )





In [13]:
if __name__ == '__main__':

    print("\nChoose Option:")
    print("1 → Run Full Pipeline")
    print("2 → Live Comment Checker")
    print("3 → Quick Tests")

    choice = input("\nEnter choice: ")

    if choice == '1':

        df_results = run_pipeline()

    elif choice == '2':

        check_comment()

    elif choice == '3':

        quick_test()

    else:

        print("Invalid choice")


Choose Option:
1 → Run Full Pipeline
2 → Live Comment Checker
3 → Quick Tests

Enter choice: 2

LIVE COMMENT CHECKER

Enter comment (or type 'exit'): I don't think the war will end anytime soon

--------------------------------------------------
Comment      : I don't think the war will end anytime soon
Sentiment    : 🔴 Negative
Emotion      : fearful or worried
Score        : 0.459
Irony        : False
Confidence   : MEDIUM
--------------------------------------------------

Enter comment (or type 'exit'): looks like the gas prices have increased but the availability has also increased

--------------------------------------------------
Comment      : looks like the gas prices have increased but the availability has also increased
Sentiment    : 🟡 Neutral
Emotion      : informed reporting
Score        : 0.412
Irony        : False
Confidence   : MEDIUM
--------------------------------------------------

Enter comment (or type 'exit'): Onion prices have tripled in Maharashtra, househol